# Import modules

In [ ]:
!pip install SciencePlots

In [ ]:
!sudo apt-get install dvipng texlive-latex-extra texlive-fonts-recommended cm-super --fix-missing

In [ ]:
import timm
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torchvision.utils import save_image
import os
import glob
from torchvision import datasets
from torch.utils.data import DataLoader
from tqdm import tqdm
import torch.optim as optim
import torch.nn.functional as F
import wandb
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
import seaborn as sns
import pandas as pd
import scienceplots

In [ ]:
plt.style.use('science')

In [ ]:
device = torch.device(torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else 'cpu')

print(device)

# Data loading

In [ ]:
#loading data
train_data_dir="/kaggle/input/datasets/kogennobori/oct-original-dataset-v3/combined3_split_rgb_jpeg/train"

test_data_dir="/kaggle/input/datasets/kogennobori/oct-original-dataset-v3/combined3_split_rgb_jpeg/test"
#Reference: https://stackoverflow.com/questions/10439104/reading-bmp-files-in-python

In [ ]:
# set image size here
img_size=224

#Reference: https://docs.pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html
train_transform = transforms.Compose([
    transforms.Resize((img_size,img_size)),
    transforms.ToTensor(),
])

tv_transform = transforms.Compose([
    transforms.Resize((img_size,img_size)),
    transforms.ToTensor(),
])
#Reference: https://stackoverflow.com/questions/69199273/torchvision-imagefolder-could-not-find-any-class-folder

train_data=datasets.ImageFolder(train_data_dir, transform=train_transform)
test_data=datasets.ImageFolder(test_data_dir,transform=tv_transform)


train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

# Testing

In [ ]:
# Loading the model. Not necessary if you already have.
model = timm.create_model('resnet34', pretrained=True, num_classes = 10)
model.to(device)
model.load_state_dict(torch.load("/kaggle/input/models/kogennobori/classifeier/pytorch/default/3/trained_net.pth"))
model.eval() # is this necessary?

In [ ]:
print(test_data.class_to_idx)

In [ ]:
# https://stackoverflow.com/questions/68047990/pytorch-imagefolder-which-index-does-a-folder-get
classes = ('AMD','CNV', 'CSR', 'DME', 'DRUSEN', 'ERM', 'NORMAL', 'RAO', 'RVO', 'VID' )

## Calculation of accuracy

In [ ]:
correct = 0
total = 0

with torch.no_grad():
  for data in test_loader:
    images,labels = data[0].to(device), data[1].to(device)
    outputs=model(images)
    _, predicted = torch.max(outputs,1)
    total+=labels.size(0)
    correct+=(predicted == labels).sum().item()

accuracy = round(100*correct/total,2)

print(f"Accuracy of the model is: {accuracy}%")

## Visualisation

In [ ]:
# https://docs.pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html
# https://stackoverflow.com/questions/59013109/runtimeerror-input-type-torch-floattensor-and-weight-type-torch-cuda-floatte
def imshow(img):
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()


dataiter = iter(test_loader)
images, labels = next(dataiter)[0].to(device), next(dataiter)[1].to(device)


# print images
imshow(torchvision.utils.make_grid(images).cpu())
print('GroundTruth: ', ' '.join(f'{classes[labels[j]]:5s}' for j in range(64)))

outputs = model(images)

_, predicted = torch.max(outputs, 1)

print('Predicted: ', ' '.join(f'{classes[predicted[j]]:5s}'
                              for j in range(64)))

## Confusion matrix

In [ ]:
# https://docs.pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html

# prepare to count predictions for each class
correct_pred = {classname: 0 for classname in classes}
total_pred = {classname: 0 for classname in classes}

# again no gradients needed
with torch.no_grad():
    for data in test_loader:
        images, labels = data[0].to(device), data[1].to(device)
        outputs = model(images)
        _, predictions = torch.max(outputs, 1)
        # collect the correct predictions for each class
        for label, prediction in zip(labels, predictions):
            if label == prediction:
                correct_pred[classes[label]] += 1
            total_pred[classes[label]] += 1


# print accuracy for each class
for classname, correct_count in correct_pred.items():
    accuracy = 100 * float(correct_count) / total_pred[classname]
    print(f'Accuracy for class: {classname:5s} is {accuracy:.1f} %')

In [ ]:
true = []
pred = []

with torch.no_grad():
    for data in test_loader:
        images, labels = data[0].to(device), data[1].to(device)
        outputs = model(images)
        _, predictions = torch.max(outputs, 1)
        # collect the correct predictions for each class
        for label, prediction in zip(labels.cpu(), predictions.cpu()):
            true.append(label)
            pred.append(prediction)

            
cm = confusion_matrix(true, pred)

# print(cm)



In [ ]:
df_cm = pd.DataFrame(cm, columns = classes, index = classes)
# df_cm = pd.DataFrame(cm)

sns.heatmap(df_cm, cmap='mako')
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.title('Confusion matrix')
plt.show()

# diverging color palette
sns.heatmap(df_cm, cmap='icefire')
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.title('Confusion matrix')
plt.show()

In [ ]:
# normalised confusion matrix
total_n = len(true)

df_cm_norm = df_cm/total_n

sns.heatmap(df_cm_norm, cmap='mako')
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.title('Confusion matrix normalised by total')
plt.show()

sns.heatmap(df_cm_norm, cmap='icefire')
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.title('Confusion matrix normalised by total')
plt.show()

In [ ]:
# https://stackoverflow.com/questions/35678874/normalize-rows-of-pandas-data-frame-by-their-sums
sums_row = np.sum(cm, axis=1)
# Source - https://stackoverflow.com/a/35679163
# Posted by Ami Tavory
# Retrieved 2026-08-25, License - CC BY-SA 3.0

df_cm_norm_rows = df_cm.div(sums_row, axis=0)

sns.heatmap(df_cm_norm_rows, cmap='mako')
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.title('Confusion matrix normalised by rows (recall on main diagonal)')
plt.show()

In [ ]:
# https://stackoverflow.com/questions/35678874/normalize-rows-of-pandas-data-frame-by-their-sums
sums_row = np.sum(cm, axis=0)


df_cm_norm_rows = df_cm.div(sums_row, axis=1)

sns.heatmap(df_cm_norm_rows, cmap='mako')
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.title('Confusion matrix normalised by columns (precision on the main diagonal)')
plt.show()

In [ ]:
# matrix summary
print(classification_report(true, pred, target_names=classes))

## Breakdown of predicted as NORMAL

In [ ]:
# classes as a list
classes = ['AMD','CNV', 'CSR', 'DME', 'DRUSEN', 'ERM', 'NORMAL', 'RAO', 'RVO', 'VID' ]

In [ ]:
bd_normal = cm[:,6]
tot_normal = bd_normal.sum()

fs=8

plt.figure(figsize=(fs,fs))
plt.bar(classes, bd_normal)
plt.title('Breakdown of images predicted as NORMAL')
plt.xlabel('True label')
plt.ylabel('Number of images')
plt.show()

plt.figure(figsize=(fs,fs))
plt.pie(bd_normal, labels=classes, startangle=90, labeldistance=None)
plt.title('Breakdown of images predicted as NORMAL')
plt.legend(title = 'True label')
plt.show()

print('Total number of images predicted as NORMAL:', tot_normal)
print('\n')
print('Breakdown of the true labels:')
for i in range(len(classes)):
    print(classes[i], round(bd_normal[i]/tot_normal*100, 2),'%')

In [ ]:
bd_normal_wout_n = np.delete(cm[:,6],6)
tot_normal_wout_n = bd_normal_wout_n.sum()

classes = ['AMD','CNV', 'CSR', 'DME', 'DRUSEN', 'ERM', 'NORMAL', 'RAO', 'RVO', 'VID' ]
classes.pop(6)
# print(bd_normal_wout_n, classes)


fs=8

plt.figure(figsize=(fs,fs))
plt.bar(classes, bd_normal_wout_n)
plt.title('Breakdown of images predicted as NORMAL (excluding true NORMAL)')
plt.xlabel('True label')
plt.ylabel('Number of images')
plt.show()

plt.figure(figsize=(fs,fs))
plt.pie(bd_normal_wout_n, labels=classes, startangle=90, labeldistance=None)
plt.title('Breakdown of images predicted as NORMAL (excluding true NORMAL)')
plt.legend(title = 'True label')
plt.show()

print('Total number of images predicted as NORMAL (excluding true NORMAL):', tot_normal_wout_n)
print('\n')
print('Breakdown of the true labels (excluding true NORMAL):')
for i in range(len(classes)):
    print(classes[i], round(bd_normal_wout_n[i]/tot_normal_wout_n*100, 2),'%')

## Saving images predicted as NORMAL

In [ ]:
classes = ['AMD','CNV', 'CSR', 'DME', 'DRUSEN', 'ERM', 'NORMAL', 'RAO', 'RVO', 'VID' ]

In [ ]:
out_dir = 'combined3_split_rgb_jpeg_predicted_NORMAL'

In [ ]:
!mkdir {out_dir}
for i in classes:
    !mkdir {out_dir}/{i}

In [ ]:
c = 0
tot_c = 0
norm_c = 0

with torch.no_grad():
    for data in test_loader:
        images, labels = data[0].to(device), data[1].to(device)
        outputs = model(images)
        _, predictions = torch.max(outputs, 1)
        # collect the correct predictions for each class
        for image, label, prediction in zip(images, labels, predictions):
            tot_c+=1
            # NORMAL..6
            # judging if the image was predicted to be NORMAL
            if prediction == 6:
                norm_c+=1
                # judging if the image is not actually NORMAL
                if label !=6:
                    c+=1
                    # image = image.cpu()
                    # imshow(image) #for visualisation. imshow is a function defined previously. This is needed as PyTorch tensor has different order of channels from the one plt accepts
                    # print('^ prediction:', classes[prediction], 'true:', classes[label])
                    # print('{}/{}/{}.jpeg'.format(out_dir, classes[label],c))
                    # saving the image
                    save_image(image, '{}/{}/{}.jpeg'.format(out_dir, classes[label],c))

print('Total number of images:', tot_c)
print('Number of images predicted as NORMAL:', norm_c)
print('Number of images falsely predicted as NORMAL:', c)
                

## Saving images predicted DME & true DME as an example

In [ ]:
out_dir_ex1 = 'combined3_split_rgb_jpeg_pred_DME_true_DME'
!mkdir {out_dir_ex1}

In [ ]:
c = 0
tot_c = 0
dme_c = 0

with torch.no_grad():
    for data in test_loader:
        images, labels = data[0].to(device), data[1].to(device)
        outputs = model(images)
        _, predictions = torch.max(outputs, 1)
        # collect the correct predictions for each class
        for image, label, prediction in zip(images, labels, predictions):
            tot_c+=1
            # DME..3
            # judging if the image was predicted to be DME
            if prediction == 3:
                dme_c+=1
                # judging if the image is actually DME
                if label == 3:
                    c+=1
                    # image = image.cpu()
                    # imshow(image) #for visualisation. imshow is a function defined previously. This is needed as PyTorch tensor has different order of channels from the one plt accepts
                    # print('^ prediction:', classes[prediction], 'true:', classes[label])
                    # print('{}/{}/{}.jpeg'.format(out_dir, classes[label],c))
                    # saving the image
                    save_image(image, '{}/{}.jpeg'.format(out_dir_ex1,c))

print('Total number of images:', tot_c)
print('Number of images predicted as DME:', dme_c)
print('Number of images for true DME:', c)
